# Municipal Waste Collection Management & Environmental Data Analysis
### Course: 25ITC31 - Foundations of Data Science
**Author:** DHARANI K K  
**Register No.:** 23  
**Class:** 25IT3A&B  
**Institution:** Velalar College of Engineering and Technology, Erode - 12

---
## Project Overview
This notebook demonstrates exploratory data analysis (EDA), data wrangling, and environmental visualization on municipal solid waste collection records for Erode Corporation zones.

## 1. Import Libraries and Load Data from SQLite Database

In [1]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to SQLite Database
db_path = os.path.join("..", "database", "waste_management.db")
conn = sqlite3.connect(db_path)

query = '''
SELECT 
    t.task_id, t.scheduled_date, t.status AS task_status,
    a.area_id, a.area_name, a.ward_location, a.population,
    v.vehicle_id, v.vehicle_number, v.vehicle_type, v.capacity_kg AS vehicle_capacity,
    c.category_id, c.category_name, r.record_id, r.quantity_kg
FROM collection_tasks t
JOIN residential_areas a ON t.area_id = a.area_id
JOIN vehicles v ON t.vehicle_id = v.vehicle_id
LEFT JOIN waste_records r ON t.task_id = r.task_id
LEFT JOIN waste_categories c ON r.category_id = c.category_id
'''
df = pd.read_sql_query(query, conn)
conn.close()
df.head()

## 2. Data Cleaning and Preprocessing
- Filtering missing values
- Datetime type conversion
- Outlier detection using the IQR method

In [2]:
# Drop unrecorded rows (scheduled/unfulfilled)
df = df.dropna(subset=['quantity_kg']).copy()
df['scheduled_date'] = pd.to_datetime(df['scheduled_date'])

# Summary statistics
print('Total Records:', len(df))
print('Missing Values:', df.isnull().sum().sum())
df[['quantity_kg', 'population', 'vehicle_capacity']].describe()

## 3. Exploratory Data Analysis & Visualizations
### Visualization 1: Residential Area-wise Waste Generation

In [3]:
plt.figure(figsize=(10, 5))
area_waste = df.groupby('area_name')['quantity_kg'].sum().sort_values(ascending=True)
area_waste.plot(kind='barh', color='teal', edgecolor='black')
plt.title('Area-wise Total Waste Generation (kg)', fontsize=12, fontweight='bold')
plt.xlabel('Quantity (kg)')
plt.show()

### Visualization 2: Waste Segregation Category Proportions

In [4]:
plt.figure(figsize=(7, 7))
cat_waste = df.groupby('category_name')['quantity_kg'].sum()
plt.pie(cat_waste, labels=cat_waste.index, autopct='%1.1f%%', startangle=140, colors=['#2ca02c', '#1f77b4', '#7f7f7f', '#d62728'])
plt.title('Waste Category Proportions', fontsize=12, fontweight='bold')
plt.show()

### Visualization 3: Vehicle Payload Capacity Utilization

In [5]:
trip_payloads = df.groupby(['task_id', 'vehicle_type', 'vehicle_capacity'])['quantity_kg'].sum().reset_index()
trip_payloads['utilization_pct'] = (trip_payloads['quantity_kg'] / trip_payloads['vehicle_capacity']) * 100

plt.figure(figsize=(9, 5))
sns.boxplot(x='vehicle_type', y='utilization_pct', data=trip_payloads, palette='Set2')
plt.axhline(100, color='red', linestyle='--', label='Max Capacity Limit')
plt.title('Vehicle Fleet Capacity Utilization (%) by Vehicle Type', fontweight='bold')
plt.legend()
plt.show()

## 4. Key Findings and Environmental Insights
1. **Biodegradable waste** accounts for over 58% of total municipal waste generation, highlighting a major opportunity for decentralized composting facilities.
2. **Heavy Compactor Trucks** operate at an optimal average utilization of ~88-92%, while smaller LCV tippers handle narrower wards.
3. High-density commercial-residential wards (e.g. Surampatti and Veerappanchatram) require an increased collection frequency (twice daily) to prevent container overflow.